This notebook uses the expanded dataset prepared by `preprocessing_refined.ipynb`: train on 2018–2024 and test on the held-out 2025 season. Run refined preprocessing before classification. The data-loading cell checks that both datasets contain the expected seasons.


In [ ]:
from pyspark.mllib.evaluation import MulticlassMetrics
from pyspark.ml.classification import DecisionTreeClassifier, LogisticRegression, RandomForestClassifier
from pyspark.ml.feature import StringIndexer
from pyspark.sql import functions as F
import matplotlib.pyplot as plt

from classification_helper import assemble, feature_importance, perform, setup_spark, print_results

In [ ]:
spark = setup_spark("Classification")

train = spark.read.parquet("./data/temporal_full_traintest/train")
test = spark.read.parquet("./data/temporal_full_traintest/test")

train_years = {row.game_year for row in train.select("game_year").distinct().collect()}
test_years = {row.game_year for row in test.select("game_year").distinct().collect()}
assert train_years == set(range(2018, 2025)), "Training data must contain the 2018-2024 seasons"
assert test_years == {2025}, "Test data must contain only the held-out 2025 season"

print("Training observations:", train.count())
print("Test observations:", test.count())
train.groupBy("game_year").count().orderBy("game_year").show()
test.groupBy("game_year").count().orderBy("game_year").show()


In [ ]:
label_indexer = StringIndexer(inputCol="pitch_type", outputCol="label")
idx_model = label_indexer.fit(train)
train = idx_model.transform(train)
test = idx_model.transform(test)

In [ ]:
features = [
    "release_speed", "release_pos_x", "release_pos_y", "release_pos_z",
    "pfx_x", "pfx_z", "plate_x", "plate_z", "vx0", "vy0",
    "vz0", "ax", "ay", "az", "effective_speed",
    "release_spin_rate", "release_extension", "spin_axis",
    "api_break_z_with_gravity", "api_break_x_arm",
    "api_break_x_batter_in", "arm_angle"
]

train, test = assemble(train, test, features)

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        featuresCol="features",
        maxIter=100
    ),
    "Decision Tree": DecisionTreeClassifier(
        featuresCol="features",
        labelCol="label",
        maxDepth=10,
        seed=42
    ),
    "Random Forest": RandomForestClassifier(
        featuresCol="features",
        labelCol="label",
        numTrees=100,
        maxDepth=10,
        seed=42
    )
}

In [ ]:
metrics = ["accuracy", "f1", "weightedPrecision", "weightedRecall"]
results = perform(train, test, models, metrics, "label")
predictions = {
    name: result["predictions"]
    for name, result in results.items()
}

In [ ]:
print_results(results)

In [ ]:
decision_tree_predictions = predictions["Decision Tree"]
pred_and_labels = (
    decision_tree_predictions
    .select("prediction", "label")
    .rdd
    .map(lambda row: (float(row.prediction), float(row.label)))
)

class_metrics = MulticlassMetrics(pred_and_labels)

print("\n")
for index, pitch in enumerate(idx_model.labels):
    print(
        f"{pitch:25s} "
        f"Precision: {class_metrics.precision(float(index)):.3f} | "
        f"Recall: {class_metrics.recall(float(index)):.3f} | "
        f"F1: {class_metrics.fMeasure(float(index)):.3f}"
    )

macro_f1 = sum(
    class_metrics.fMeasure(float(index))
    for index in range(len(idx_model.labels))
) / len(idx_model.labels)
print(f"Macro F1: {macro_f1:.4f}")

In [ ]:
random_forest = models["Random Forest"].fit(train)
rf_importance = feature_importance(random_forest, train)
print(rf_importance.head(30).to_string(index=False))

top_features = rf_importance.head(20).sort_values("importance")
plt.figure(figsize=(10, 8))
plt.barh(top_features["feature"], top_features["importance"])
plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title("Random Forest - Top 20 Feature Importances")
plt.tight_layout()
plt.show()